# 1104 44일차

## 1. 양자화 파이프라인

허깅페이스에 올라온 원본 모델을 낮은 비트로 줄여서, 내 환경에서 돌릴 수 있는 파일로 만드는 과정

원본 가중치는 BF16 같은 16비트 실수라 용량과 메모리를 많이 씀. 비트 수를 줄이면 파일이 작아지고 메모리에 덜 올라가서 GPU 없이 CPU만으로도 모델을 띄울 수 있음

![원본 safetensors를 변환해 BF16 GGUF를 만들고 다시 양자화해 Q4_K_M GGUF를 만드는 흐름. 변환에서는 크기가 거의 그대로고 양자화에서 3분의 1로 줄어든다](assets/quantization-pipeline.svg)

1. 원본 : 허깅페이스 저장소에 올라온 BF16 가중치
2. 변환 : BF16 → GGUF. 자료형은 그대로 두고 파일 형식만 바꿈
3. 양자화 : GGUF → Q4_K_M. 비트 수를 줄임

- 변환에서는 크기가 거의 그대로고 양자화에서 줄어듦. 두 단계가 서로 다른 일을 한다는 표시
- GGUF는 llama.cpp가 쓰는 모델 파일 형식. 여러 개로 나뉜 가중치와 토크나이저·메타데이터를 한 파일에 담음
- BF16은 16비트 실수. 지수부는 32비트 실수와 같고 소수부만 짧아서, 표현 범위는 유지하고 정밀도만 줄인 형태
- BPW(bits per weight)는 가중치 하나당 평균 비트 수. 양자화 전후를 비교하는 기준
- 한 번 만든 BF16 GGUF를 두고 타입만 바꿔 여러 번 양자화할 수 있어서, 중간 파일을 지우지 않고 둠

## 2. Ollama

명령 한 줄로 LLM을 받아서 실행하는 도구

모델을 내려받고 실행 환경을 맞추는 과정이 한 명령에 묶여 있어서, 설정 없이 모델을 먼저 써 볼 수 있음

1. `wsl --install -d Ubuntu-24.04 --name quant` : 양자화 작업용 WSL 배포판을 따로 만듦
2. `curl -fsSL https://ollama.com/install.sh | sh` : 설치 스크립트를 받아서 바로 셸로 실행
3. `systemctl status ollama` : ollama 서비스가 떠 있는지 확인
4. `ollama run gemma4:e2b` : 모델을 받아서 대화 프롬프트로 들어감
5. `ollama list` : 받아 둔 모델 목록과 용량

- `quant`는 내가 정한 배포판 이름. 작업마다 배포판을 나누면 패키지가 섞이지 않음
- `curl`의 `-f`는 실패하면 에러로 끝내기, `-s`는 진행률 숨기기, `-S`는 에러만 보여주기, `-L`은 리다이렉트 따라가기. `| sh`는 받은 내용을 파일로 저장하지 않고 바로 셸에 넘겨 실행
- 설치 스크립트가 ollama를 systemd 서비스로 등록해서, 따로 띄우지 않아도 `systemctl`로 상태를 봄
- `gemma4:e2b`의 `:` 뒤는 태그. 모델 이름은 같고 크기나 양자화가 다른 것들을 태그로 구분함
- `ollama run`에 적는 것은 파일 이름이 아니라 등록된 모델 이름. 없는 이름을 적으면 인터넷 저장소에서 찾으려다 실패함
- `ollama create`는 GGUF를 ollama 저장소로 복사함. 등록할 때마다 같은 용량이 한 벌 더 생김
- ollama로는 양자화를 할 수 없어서, 양자화하려면 llama.cpp를 받아 빌드해야 함

## 3. llama.cpp 빌드

GGUF 변환과 양자화 도구가 들어 있는 저장소를 받아서 직접 컴파일하는 단계

llama.cpp는 설치 패키지로 배포되지 않고 소스로 공개돼 있어서, 받아서 빌드해야 `llama-quantize` 같은 실행 파일이 생김

1. `sudo apt install -y cmake build-essential python3-pip` : 빌드 도구 설치
2. `git clone https://github.com/ggml-org/llama.cpp.git` : 저장소 받기
3. `cmake -B build -DCMAKE_BUILD_TYPE=Release` : 빌드 설정을 `build` 폴더에 만듦
4. `cmake --build build --config Release -j 8` : 실제 컴파일
5. `./build/bin/llama-quantize --help` : 빌드된 실행 파일과 지원하는 양자화 타입 확인

- `cmake`는 빌드 설정 도구, `build-essential`은 컴파일러와 make 묶음, `python3-pip`는 변환 스크립트용
- `-B build` : 빌드 파일을 둘 폴더. `build`는 관례로 쓰는 이름이라 바꿔도 됨
- `-D` : cmake 변수 지정. `CMAKE_BUILD_TYPE=Release`는 최적화를 켜고 디버그 정보를 빼서 실행이 빠름
- `-j 8` : 동시에 돌릴 컴파일 작업 개수. 코어 수에 맞춰 내가 정함
- 빌드가 끝나면 실행 파일이 `build/bin` 아래에 생김. `./`를 붙이는 것은 현재 폴더의 파일을 실행한다는 뜻
- 새로 나온 아키텍처는 변환 지원이 늦게 들어오기도 해서, 받아 둔 지 오래됐으면 `git pull` 후 다시 빌드함

## 4. 파이썬 가상환경

프로젝트 전용 파이썬 환경을 만들어서, 변환 스크립트가 쓰는 패키지를 거기에만 설치하는 것

변환 스크립트는 토치와 트랜스포머를 `import`하는데 시스템 파이썬에는 없음. 우분투 24.04는 시스템 파이썬에 pip로 바로 설치하는 것을 막아 둬서 환경을 따로 만들어야 함

1. `sudo apt install -y python3-venv` : venv 모듈 설치
2. `python3 -m venv ~/gemma4-quant/.venv` : 가상환경 생성
3. `source ~/gemma4-quant/.venv/bin/activate` : 활성화
4. `python -m pip install -r requirements/requirements-convert_hf_to_gguf.txt` : 변환 스크립트가 쓰는 패키지만 설치
5. `python -c "import torch; print(torch.__version__)"` : 설치 확인

- `.venv` 경로와 이름은 내가 정함. 점으로 시작하는 이름은 숨김 파일이라 `ls`에 안 보임
- 활성화하면 프롬프트 앞에 `(.venv)`가 붙고 `python`·`pip`가 가상환경 것을 가리킴
- `python -m pip`처럼 `-m`을 쓰면 지금 활성화된 파이썬의 pip가 돌아감
- conda와 하는 일은 같지만 venv는 파이썬에 들어 있는 기능이라 따로 설치할 것이 적음
- `source`는 현재 셸의 `PATH`를 바꾸는 명령이라 터미널을 새로 열면 풀림. **가상환경은 셸 세션마다 다시 활성화해야 함**

#### cf) requirements.txt가 목록만 들어 있을 때

- llama.cpp의 `requirements.txt`에는 패키지 이름이 없고 `-r`로 다른 파일을 불러오는 줄만 들어 있음
- 실제 패키지 이름은 `requirements/` 아래에 스크립트별로 나뉘어 있어서, 쓸 스크립트 것만 골라 설치하면 됨
- 불러오는 파일 쪽에 `==`로 버전이 박혀 있으면, 수동으로 올려 둔 패키지도 requirements를 다시 설치할 때 그 버전으로 되돌아감
- **패키지를 수동으로 올린 뒤에는 requirements를 다시 설치하지 않음**
- `python -m pip check` : 설치된 패키지들의 의존성이 서로 맞는지 확인

## 5. GGUF 변환

허깅페이스 형식의 원본 가중치를 GGUF 파일로 바꾸는 단계

`llama-quantize`는 GGUF만 입력으로 받음. 원본은 허깅페이스 형식이라 먼저 GGUF로 바꿔야 양자화할 수 있음

```
python3 convert_hf_to_gguf.py ~/gemma4-quant/Qwen3-4B \
  --outfile ../qwen3-4b-bf16.gguf \
  --outtype bf16
```

1. 첫 자리 : 읽을 원본 폴더
2. `--outfile` : 만들 GGUF 파일 경로
3. `--outtype bf16` : GGUF에 저장할 자료형

- 줄 끝의 `\`는 명령이 다음 줄로 이어진다는 표시. 옵션이 많을 때 줄을 나눠 씀
- `../qwen3-4b-bf16.gguf`의 `..`는 llama.cpp 바로 위인 작업 폴더. 파일 이름은 내가 정함
- `--outtype`을 `bf16`으로 둬서 원본과 같은 자료형을 유지함. 정밀도를 줄이는 것은 양자화 단계에서 함
- `--remote <계정>/<모델>`을 쓰면 내려받지 않고 허깅페이스에서 바로 읽음. 모델 이름 뒤의 `-it`는 instruction tuned
- **변환이 에러 없이 끝났다고 결과가 맞는 것은 아님.** 모델을 모르는 버전으로 변환하면 파일은 만들어지고 내용만 깨짐
- 변환이 잘못되면 양자화 결과도 같이 깨져서 뒤에서는 어느 단계 탓인지 모름. 양자화 전에 변환 결과를 먼저 돌려서 확인함

#### cf) 변환이 막힐 때

- `ModuleNotFoundError: No module named 'torch'` : 가상환경이 활성화되지 않았거나 패키지가 설치되지 않은 상태
- `AttributeError: 'list' object has no attribute 'keys'` : 설치된 트랜스포머가 그 모델의 설정 형식을 모를 때 남. `python -m pip install --upgrade transformers`로 올림
- 새로 나온 모델은 변환 스크립트와 라이브러리 양쪽 모두 그 모델을 아는 버전이어야 함
- 같은 모델이라도 `--remote`보다 내려받아서 변환하는 쪽이 덜 막힘

#### cf) 디스크가 꽉 찰 때

- WSL은 `ext4.vhdx`라는 가상 디스크 파일 하나를 늘려가며 씀. 윈도우 쪽 드라이브에 여유가 없으면 더 늘릴 수 없어 읽기·쓰기가 `Input/output error`로 떨어짐
- 실행 파일 자체를 디스크에서 못 읽어서 `ls` 같은 기본 명령까지 같이 실패함. 파일이 깨진 것이 아니라 공간이 없는 것
- 탐색기로 지우면 휴지통으로 옮겨질 뿐이라 공간이 비지 않음. `Shift + Delete`로 지우거나 휴지통을 비워야 함
- WSL 안에서 파일을 지워도 `ext4.vhdx` 크기는 그대로임. **가상 디스크는 늘어나기만 하고 자동으로 줄지 않음**
- `wsl --shutdown` 후 `wsl --manage <배포판> --set-sparse true` : 가상 디스크의 빈 공간을 윈도우에 돌려줌

## 6. 양자화 실행

BF16 GGUF의 가중치를 4비트로 줄여서 새 GGUF를 만드는 단계

변환 스크립트의 `--outtype`은 `f32`·`f16`·`bf16`·`q8_0` 같은 단순한 자료형만 받음. 블록마다 기준값을 두는 K-quant는 `llama-quantize`만 할 줄 알아서 단계를 나눔

```
./build/bin/llama-quantize \
../qwen3-4b-bf16.gguf \
../qwen3-4b-Q4_K_M.gguf \
Q4_K_M
```

1. 첫 번째 인자 : 입력 GGUF
2. 두 번째 인자 : 만들 GGUF 파일 경로
3. 세 번째 인자 : 양자화 타입

- 옵션 이름 없이 순서로만 구분함. **순서를 바꾸면 입력 파일을 덮어씀**
- `Q4_K_M`의 `Q4`는 4비트, `K`는 블록마다 기준값을 따로 두는 K-quant 방식, `M`은 그 안에서 중간 크기
- K-quant는 가중치를 작은 블록으로 나눠 블록마다 기준값을 두므로, 전체를 똑같이 4비트로 깎는 것보다 정밀도가 덜 떨어짐
- 임베딩처럼 비트를 덜 줄이는 텐서가 섞여 있어서, 결과 BPW는 타입 이름의 4비트보다 크게 나옴
- 만든 GGUF를 ollama에 등록하려면 `FROM <GGUF 경로>`를 적은 Modelfile을 만들고 `ollama create <이름> -f <Modelfile>`

#### cf) 모델이 이상한 답을 할 때 어느 단계인지 가리기

- `<unused32>` 같은 특수 토큰만 계속 나오면 가중치·토크나이저·대화 템플릿 중 하나가 안 맞는 상태
- `./build/bin/llama-completion -m <GGUF> -p "<문장>"` : ollama를 빼고 GGUF만 돌려봄. 여기서도 깨지면 ollama 설정 문제가 아님
- 양자화 결과와 변환 결과(BF16)를 각각 돌려봄. BF16도 깨지면 양자화가 아니라 변환 단계 문제
- 같은 llama.cpp 바이너리로 공식 GGUF를 돌려보면, 내 변환물 문제인지 llama.cpp가 그 아키텍처를 못 돌리는 것인지 갈림
- `python gguf-py/gguf/scripts/gguf_dump.py --no-tensors <GGUF>` : 메타데이터만 출력. 지시 학습 모델을 제대로 변환하면 `tokenizer.chat_template`이 들어가 있음
- GGUF에 `chat_template`이 있으면 llama.cpp가 대화 모드로 바뀜. Jinja 형식이면 `--jinja`를 붙여야 읽음
- `ollama show <모델> --modelfile` : 내가 등록한 설정과 공식 모델의 설정을 비교함